# Simple SDP 2026 Ward + LAD Code Matcher

This notebook fills:

- `WD26CD` from the 2026 ward names-and-codes lookup: `WD_MAY_2026_UK_NC.csv`
- `LAD26CD` from the April 2025 LAD names-and-codes lookup: `Local_Authority_Districts_(April_2025)_Names_and_Codes_in_the_UK_v2.csv`

The LAD source is a 2025 lookup, but the results output keeps the user's `LAD26CD` column naming convention.

Important limitation: the ward file is a names-and-codes file only. It does not contain LAD codes, so duplicate ward names cannot be resolved safely by LAD unless you provide a separate `WD26 -> LAD` lookup or add a manual override.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata

DATA_DIR = Path(".")  # Change this if your CSVs are somewhere else.

RAW_RESULTS = DATA_DIR / "sdp_candidate_results_2026_raw_v1.csv"
WARD_LOOKUP = DATA_DIR / "WD_MAY_2026_UK_NC.csv"
LAD_LOOKUP = DATA_DIR / "Local_Authority_Districts_(April_2025)_Names_and_Codes_in_the_UK_v2.csv"

OUTPUT_MATCHED = DATA_DIR / "sdp_candidate_results_2026_with_ward_and_lad_codes_v2.csv"
OUTPUT_REVIEW = DATA_DIR / "sdp_candidate_results_2026_unmatched_or_ambiguous_ward_lad_v2.csv"


## Helper functions

These functions keep the notebook tolerant of small naming differences, including files where the results columns are still named `WD25*` / `LAD25*` rather than `WD26*` / `LAD26*`.


In [2]:
def normalise_name(value):
    """Normalise names for safer exact joins."""
    if pd.isna(value):
        return ""
    value = str(value).strip()
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    value = value.lower()
    value = value.replace("&", " and ")
    value = re.sub(r"['’`]", "", value)
    value = re.sub(r"[-/]", " ", value)
    value = re.sub(r"[^a-z0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def find_column(df, candidates):
    """Return the first available column from a candidate list."""
    for col in candidates:
        if col in df.columns:
            return col
    raise KeyError(
        f"None of these columns were found: {candidates}\n"
        f"Available columns: {df.columns.tolist()}"
    )


## Load files and standardise result column names

The output is standardised to `WD26NM`, `WD26CD`, `LAD26NM`, and `LAD26CD`, even if the input file currently uses `25` column names.


In [3]:
results_raw = pd.read_csv(RAW_RESULTS, dtype=str)
wards_raw = pd.read_csv(WARD_LOOKUP, dtype=str)
lads_raw = pd.read_csv(LAD_LOOKUP, dtype=str)

raw_wd_name_col = find_column(results_raw, ["WD26NM", "WD25NM", "Ward", "Ward Name", "ward_name"])
raw_wd_code_col = find_column(results_raw, ["WD26CD", "WD25CD", "Ward Code", "ward_code"])
raw_lad_name_col = find_column(results_raw, ["LAD26NM", "LAD25NM", "Local Authority", "Council", "LAD Name", "lad_name"])
raw_lad_code_col = find_column(results_raw, ["LAD26CD", "LAD25CD", "LAD Code", "lad_code"])

print("Detected result columns:")
print("Ward name:", raw_wd_name_col)
print("Ward code:", raw_wd_code_col)
print("LAD name:", raw_lad_name_col)
print("LAD code:", raw_lad_code_col)

results = results_raw.rename(columns={
    raw_wd_name_col: "WD26NM",
    raw_wd_code_col: "WD26CD",
    raw_lad_name_col: "LAD26NM",
    raw_lad_code_col: "LAD26CD",
}).copy()

results.head()


Detected result columns:
Ward name: WD26NM
Ward code: WD26CD
LAD name: LAD26NM
LAD code: LAD26CD


,Election Year,Election Type,Area Type,WD26NM,WD26CD,Authority Type,LAD26NM,LAD26CD,Candidate,Incumbent?,Votes,Votes (Perc.),Turnout,Turnout (Perc.),Place,No. Candidates,Seats Available,Total Electors,Previous Election,Previous Winner
0,2026,Locals,Ward,Dearne South,NaN,NaN,Barnsley,NaN,Warwick Bettney,NaN,70,2.60%,2722,28.89,13,NaN,NaN,9422,NaN,NaN
1,2026,Locals,Ward,Dearne South,NaN,NaN,Barnsley,NaN,David Allan Jarvis,NaN,111,4.10%,2722,28.89,9,NaN,NaN,9422,NaN,NaN
2,2026,Locals,Ward,Penistone West,NaN,NaN,Barnsley,NaN,Rori Cook,NaN,180,3.80%,4767,48.69,16,NaN,NaN,9790,NaN,NaN
3,2026,Locals,Ward,Bingley East,NaN,NaN,Bradford,NaN,Alexander Richard Vann,NaN,42,0.20%,20196,NaN,15,NaN,NaN,13507,NaN,NaN
4,2026,Locals,Ward,Bingley West,NaN,NaN,Bradford,NaN,Paul Shkurka,NaN,70,0.40%,17933,NaN,14,NaN,NaN,12792,NaN,NaN


## Prepare lookup tables

For safety, the notebook collapses each lookup by normalised name and records how many possible code matches exist.

A ward is filled automatically only where the ward name has exactly one code in the 2026 ward lookup.

A LAD is filled automatically only where the LAD name has exactly one code in the 2025 LAD lookup.


In [4]:
results["_ward_name_key"] = results["WD26NM"].map(normalise_name)
results["_lad_name_key"] = results["LAD26NM"].map(normalise_name)

wards = wards_raw[["WD26CD", "WD26NM"]].copy()
wards["_ward_name_key"] = wards["WD26NM"].map(normalise_name)

lads = lads_raw[["LAD25CD", "LAD25NM"]].copy()
lads["_lad_name_key"] = lads["LAD25NM"].map(normalise_name)

ward_lookup_by_name = (
    wards.groupby("_ward_name_key", dropna=False)
    .agg(
        ward_match_count=("WD26CD", "size"),
        ward_candidate_codes=("WD26CD", lambda s: "; ".join(sorted(s.dropna().astype(str).unique()))),
        ward_candidate_names=("WD26NM", lambda s: "; ".join(sorted(s.dropna().astype(str).unique()))),
    )
    .reset_index()
)

lad_lookup_by_name = (
    lads.groupby("_lad_name_key", dropna=False)
    .agg(
        lad_match_count=("LAD25CD", "size"),
        lad_candidate_codes=("LAD25CD", lambda s: "; ".join(sorted(s.dropna().astype(str).unique()))),
        lad_candidate_names=("LAD25NM", lambda s: "; ".join(sorted(s.dropna().astype(str).unique()))),
    )
    .reset_index()
)

print("Ward lookup rows:", len(wards_raw))
print("Unique normalised ward names:", ward_lookup_by_name["_ward_name_key"].nunique())
print("LAD lookup rows:", len(lads_raw))
print("Unique normalised LAD names:", lad_lookup_by_name["_lad_name_key"].nunique())


Ward lookup rows: 8413
Unique normalised ward names: 7921
LAD lookup rows: 361
Unique normalised LAD names: 361


## Match results to ward and LAD codes

In [5]:
matched = (
    results
    .merge(ward_lookup_by_name, on="_ward_name_key", how="left")
    .merge(lad_lookup_by_name, on="_lad_name_key", how="left")
)

matched["WD26CD"] = np.where(
    matched["ward_match_count"].eq(1),
    matched["ward_candidate_codes"],
    matched["WD26CD"]
)

# The LAD source is LAD25, but the output column is kept as LAD26CD by convention.
matched["LAD26CD"] = np.where(
    matched["lad_match_count"].eq(1),
    matched["lad_candidate_codes"],
    matched["LAD26CD"]
)

matched["ward_code_match_status"] = np.select(
    [
        matched["ward_match_count"].eq(1),
        matched["ward_match_count"].gt(1),
        matched["ward_match_count"].isna(),
    ],
    [
        "matched_unique_ward_name",
        "ambiguous_ward_name",
        "unmatched_ward_name",
    ],
    default="check_ward_match",
)

matched["lad_code_match_status"] = np.select(
    [
        matched["lad_match_count"].eq(1),
        matched["lad_match_count"].gt(1),
        matched["lad_match_count"].isna(),
    ],
    [
        "matched_unique_lad_name_from_lad25_lookup",
        "ambiguous_lad_name_from_lad25_lookup",
        "unmatched_lad_name_from_lad25_lookup",
    ],
    default="check_lad_match",
)

matched["overall_code_match_status"] = np.where(
    matched["ward_code_match_status"].eq("matched_unique_ward_name")
    & matched["lad_code_match_status"].eq("matched_unique_lad_name_from_lad25_lookup"),
    "matched",
    "review_required",
)

matched["match_note"] = ""

matched.loc[matched["ward_code_match_status"].eq("ambiguous_ward_name"), "match_note"] = (
    "Ward name appears multiple times in WD26 names-and-codes lookup. "
    "A WD26-to-LAD lookup or manual check is required."
)

matched.loc[matched["ward_code_match_status"].eq("unmatched_ward_name"), "match_note"] = (
    "Ward name not found as an exact WD26 ward name. "
    "Check whether this is a county division, renamed ward, or composite area."
)

matched.loc[matched["lad_code_match_status"].eq("unmatched_lad_name_from_lad25_lookup"), "match_note"] = (
    matched["match_note"].where(matched["match_note"].eq(""), matched["match_note"] + " ")
    + "Authority name not found in LAD25 lookup. "
    "Check whether this is a county/non-LAD authority name."
)

matched[[
    "WD26NM", "WD26CD", "LAD26NM", "LAD26CD",
    "ward_code_match_status", "lad_code_match_status", "overall_code_match_status"
]].head(10)


,WD26NM,WD26CD,LAD26NM,LAD26CD,ward_code_match_status,lad_code_match_status,overall_code_match_status
0,Dearne South,E05016574,Barnsley,E08000038,matched_unique_ward_name,matched_unique_lad_name_from_lad25_lookup,matched
1,Dearne South,E05016574,Barnsley,E08000038,matched_unique_ward_name,matched_unique_lad_name_from_lad25_lookup,matched
2,Penistone West,E05016582,Barnsley,E08000038,matched_unique_ward_name,matched_unique_lad_name_from_lad25_lookup,matched
3,Bingley East,E05016455,Bradford,E08000032,matched_unique_ward_name,matched_unique_lad_name_from_lad25_lookup,matched
4,Bingley West,E05016456,Bradford,E08000032,matched_unique_ward_name,matched_unique_lad_name_from_lad25_lookup,matched
5,"Cropredy, Sibfords & Wroxton",E05010929,Cherwell,E07000177,matched_unique_ward_name,matched_unique_lad_name_from_lad25_lookup,matched
6,Earlsdon,E05016401,Coventry,E08000026,matched_unique_ward_name,matched_unique_lad_name_from_lad25_lookup,matched
7,Earlsdon,E05016401,Coventry,E08000026,matched_unique_ward_name,matched_unique_lad_name_from_lad25_lookup,matched
8,Earlsdon,E05016401,Coventry,E08000026,matched_unique_ward_name,matched_unique_lad_name_from_lad25_lookup,matched
9,Hanger Hill,E05013524,Ealing,E09000009,matched_unique_ward_name,matched_unique_lad_name_from_lad25_lookup,matched


## Optional manual overrides

Use this only after checking the correct code manually. The key format is `(LAD name, Ward name)`. Leave it empty if you want a strict automatic match only.


In [ ]:
WARD_MANUAL_OVERRIDES = {
    # Example:
    # ("Reading", "Thames"): "E05000000",
}

LAD_MANUAL_OVERRIDES = {
    # Example:
    # ("Some Non-LAD Authority Name"): "E00000000",
}

for (lad_name, ward_name), ward_code in WARD_MANUAL_OVERRIDES.items():
    mask = (
        matched["LAD26NM"].map(normalise_name).eq(normalise_name(lad_name))
        & matched["WD26NM"].map(normalise_name).eq(normalise_name(ward_name))
    )
    matched.loc[mask, "WD26CD"] = ward_code
    matched.loc[mask, "ward_code_match_status"] = "matched_manual_override"

for lad_name, lad_code in LAD_MANUAL_OVERRIDES.items():
    mask = matched["LAD26NM"].map(normalise_name).eq(normalise_name(lad_name))
    matched.loc[mask, "LAD26CD"] = lad_code
    matched.loc[mask, "lad_code_match_status"] = "matched_manual_override"

matched["overall_code_match_status"] = np.where(
    matched["WD26CD"].notna()
    & matched["LAD26CD"].notna()
    & ~matched["WD26CD"].astype(str).str.strip().eq("")
    & ~matched["LAD26CD"].astype(str).str.strip().eq(""),
    "matched",
    "review_required",
)


## Export matched file and review file

In [6]:
helper_cols = [
    "ward_match_count", "ward_candidate_codes", "ward_candidate_names",
    "lad_match_count", "lad_candidate_codes", "lad_candidate_names",
    "ward_code_match_status", "lad_code_match_status",
    "overall_code_match_status", "match_note"
]

core_cols = [
    c for c in matched.columns
    if c not in helper_cols + ["_ward_name_key", "_lad_name_key"]
]

matched_export = matched[core_cols + helper_cols].copy()
review = matched_export[matched_export["overall_code_match_status"].ne("matched")].copy()

matched_export.to_csv(OUTPUT_MATCHED, index=False)
review.to_csv(OUTPUT_REVIEW, index=False)

print("Rows in input:", len(results_raw))
print("Rows matched:", (matched_export["overall_code_match_status"] == "matched").sum())
print("Rows needing review:", (matched_export["overall_code_match_status"] == "review_required").sum())
print()
print("Saved:", OUTPUT_MATCHED)
print("Saved:", OUTPUT_REVIEW)

review[[
    "WD26NM", "WD26CD", "LAD26NM", "LAD26CD",
    "ward_code_match_status", "lad_code_match_status",
    "ward_candidate_codes", "match_note"
]]


Rows in input: 48
Rows matched: 44
Rows needing review: 4

Saved: sdp_candidate_results_2026_with_ward_and_lad_codes_v2.csv
Saved: sdp_candidate_results_2026_unmatched_or_ambiguous_ward_lad_v2.csv


,WD26NM,WD26CD,LAD26NM,LAD26CD,ward_code_match_status,lad_code_match_status,ward_candidate_codes,match_note
10,Dorking,NaN,East Surrey,NaN,unmatched_ward_name,unmatched_lad_name_from_lad25_lookup,NaN,Ward name not found as an exact WD26 ward name...
11,"Esher, Claygate & Oxshott North",NaN,East Surrey,NaN,unmatched_ward_name,unmatched_lad_name_from_lad25_lookup,NaN,Ward name not found as an exact WD26 ward name...
12,Catherington,NaN,Hampshire,NaN,unmatched_ward_name,unmatched_lad_name_from_lad25_lookup,NaN,Ward name not found as an exact WD26 ward name...
43,Thames,NaN,Reading,E06000038,ambiguous_ward_name,matched_unique_lad_name_from_lad25_lookup,E05011718; E05013877; E05015796,Ward name appears multiple times in WD26 names...
